In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
from pathlib import Path

notebook_path = "/u/skarmakar1/version_check/llm_steering-main/sk"
sys.path.append(notebook_path)

In [3]:
import torch
import numpy as np

from inversion_utils import *
import pickle
from sklearn.model_selection import train_test_split

In [4]:
SEED = 0
# SEED = 1

torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
np.random.seed(SEED)

torch.backends.cudnn.benchmark = True 
torch.backends.cuda.matmul.allow_tf32 = True

LLM = namedtuple('LLM', ['language_model', 'tokenizer', 'processor', 'name', 'model_type'])

In [5]:
model_type = 'llama'
MODEL_VERSION='3.1'
MODEL_SIZE='8B'

# model_type = 'gemma'
# # MODEL_VERSION='2'
# MODEL_VERSION='3'
# # MODEL_SIZE='1B'
# # MODEL_SIZE='9B'
# MODEL_SIZE='12B'

# model_type = 'qwen'
# MODEL_VERSION='3'
# MODEL_SIZE='4B'
# # MODEL_SIZE='8B'


llm = select_llm(model_type, MODEL_VERSION=MODEL_VERSION, MODEL_SIZE=MODEL_SIZE)

Loading meta-llama/Meta-Llama-3.1-8B-Instruct


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [6]:
hidden_layers = list(range(-1, -llm.language_model.config.num_hidden_layers, -1))
print(hidden_layers)

[-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]


In [7]:
all_langs = ['english', 'german', 'hindi', 'spanish', 'french', 'italian', 'portuguese', 'thai', 'chinese_simplified', 'chinese_traditional', 'japanese', 'korean', 'russian', 'arabic', 'vietnamese', 'indonesian', 'turkish', 'polish', 'dutch', 'ukrainian', 'czech', 'romanian', 'greek', 'hungarian', 'swedish', 'danish', 'finnish', 'norwegian', 'bulgarian', 'serbian', 'croatian', 'slovak', 'lithuanian', 'slovenian', 'latvian', 'estonian', 'catalan', 'hebrew', 'persian', 'tagalog', 'bengali', 'urdu', 'tamil', 'telugu', 'malayalam', 'kannada', 'marathi', 'gujarati', 'punjabi', 'malay', 'swahili', ]

In [10]:
# coef = 0.65
coef = 0.7 # llama
# coef = 0.75

# coef = 2.0
# coef = 2.5 # qwen
# coef = 3.0
# coef = 3.5

# coef = 65.0
# coef = 80.0


path = "../all_gitignore/xRFM/language_multiex_llama/"
# path = "../all_gitignore/xRFM/language_multi_gemma"
# path = "../all_gitignore/xRFM/language_multi_qwen"
# path = "../all_gitignore/xRFM/language_multiex_qwen"
# path = "../all_gitignore/xRFM/test"

max_tokens = 200
# max_tokens = 100

# prompts = ["How does a clock work?"] # "english"
prompts = ["What is weight of a football used in fifa?"] # "english"
# prompts = ["Quel est le poids d'un ballon de football utilisé par la FIFA ?"] # "french"
# prompts = ["फीफा में इस्तेमाल होने वाली फुटबॉल का वज़न कितना होता है?"] # "hindi"
# prompts = ["¿Cuál es el peso de un balón de fútbol utilizado en la FIFA?"] # 'spanish'


orig_lang = "english"
# orig_lang = "hindi"
# orig_lang = "german"
# orig_lang = 'spanish'
# orig_lang = 'french'

# l = "english"
# l = "german"
# l = "hindi"
# l = "thai"
# l = 'italian'
# l = 'french'
# l = 'portuguese'

l = "czech"

l_controller = load_controller_translate(llm, l, orig_lang, path=path)

out = test_concept_vector(l_controller, concept=f"coef: {coef}, lang: {l}", prompts=prompts, coef=coef, max_tokens=max_tokens, qwen=False, orig=False)
# out = test_concept_vector(l_controller, concept=f"coef: {coef}, lang: {l}", prompts=prompts, coef=coef, max_tokens=max_tokens, qwen=True, orig=False)

# for l in all_langs:
#     if l == orig_lang or l == "italian":
#         continue

#     # l_controller = load_controller(llm, l, path=f'../all_gitignore/language_multi/')
#     l_controller = load_controller_translate(llm, l, orig_lang, path=path)

#     # out = test_concept_vector(l_controller, concept=f"coef: {coef}, lang: {l}", prompts=prompts, coef=coef, max_tokens=max_tokens, qwen=False, orig=False)
#     out = test_concept_vector(l_controller, concept=f"coef: {coef}, lang: {l}", prompts=prompts, coef=coef, max_tokens=max_tokens, qwen=True, orig=False)

Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 8
forward_batch_size   : 16
M_batch_size         : 2048
n_components         : 1

Using xRFM library for RFM-based steering vector extraction
Detector found

========================== + coef: 0.7, lang: czech Control (normal) ==========================
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

What is weight of a football used in fifa?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Fotbal používaný v rámci FIFA je poměrně těžký, ale existují i některé zajímavé informace, které by mohly být pro vás zajímavé. 

Fotbal používaný v rámci FIFA má hmotnost přibližně 450 g. To je ovšem pouze přibližná hodnota a můž

In [1]:
# LLama

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

custom_cache_dir = "/scratch/bbjr/skarmakar/huggingface"

model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token_id = 0 
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    device_map="auto", 
    cache_dir=custom_cache_dir,
    
)

model.generation_config.pad_token_id = tokenizer.pad_token_id

model.generation_config.temperature=None # to disable the stupid warnings
model.generation_config.top_p=None # to disable the stupid warnings
model.generation_config.top_k=None # to disable the stupid warnings

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [2]:
messages = [
    {"role": "user", "content": "国际足联比赛中使用的足球的重量是多少？"}
]

# # Apply the template and move to device
# inputs = tokenizer.apply_chat_template(
#     messages, 
#     add_generation_prompt=True, 
#     return_tensors="pt"
# ).to(model.device)

# outputs = model.generate(inputs, max_new_tokens=256, do_sample=False)
# print(tokenizer.decode(outputs[0], skip_special_tokens=True))

prompts = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
# prompts = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True).strip()

out = tokenizer(prompts, return_tensors='pt', padding=True, add_special_tokens=False).to(model.device)

outputs = model.generate(**out, max_new_tokens=300, do_sample=False)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

system

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

user

国际足联比赛中使用的足球的重量是多少？assistant

国际足联比赛中使用的足球的重量是420-450克。


content: **Футбольний м'яч, який використовується у ФІФА**, має **вагу від 410 грамів до 450 грамів**.

Це встановлено в офіційних правилах ФІФА (Fédération Internationale de Football Association). Вага м'яча повинна бути **в межах 410–450 грамів**, а його об'єм і форма також регулюються для забезпечення однакових умов гри.

👉 Також важливо, що м'яч має бути **сферичним**, з **нормальною твердістю** і **відповідною гладкістю поверхні**, щоб забезпечити рівність у гравіні.

✅ Важливо: це вага **навантаженого м'яча** (з урахуванням всіх елементів, включаючи оболонку та наповнення), а не вага порожнього.

**Відповідь


In [10]:
# prompt = "What is weight of a football used in fifa?"
prompt = "फीफा में इस्तेमाल होने वाली फुटबॉल का वज़न कितना होता है?"

messages = [
    {"role": "user", "content": prompt}
]
text = llm.tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
model_inputs = llm.tokenizer([text], return_tensors="pt").to(llm.language_model.device)

# conduct text completion
generated_ids = llm.language_model.generate(
    **model_inputs,
    max_new_tokens=16384
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

content = llm.tokenizer.decode(output_ids, skip_special_tokens=True)

print("content:", content)

content: फीफा (FIFA) के नियमों के अनुसार, फुटबॉल (फिटबॉल) के अंतर्गत इस्तेमाल होने वाले फुटबॉल का **वजन 410 ग्राम से 450 ग्राम के बीच** होता है।

इसके अलावा, फीफा के नियम के अनुसार फुटबॉल के विशेष आवश्यक गुण इनके अंतर्गत शामिल हैं:

- **विस्तार** (पैरामीटर): 68 से 70 सेमी (लंबाई)
- **व्यास**: 22 से 23 सेमी
- **वजन**: 410 ग्राम से 450 ग्राम

इन मापदोतर को रखने का उद्देश्य इसकी आम खेल की रूचि, प्रतिस्पर्धा और सभी खिलाड़ियों के लिए समानता बनाए रखना है।

✅ इसलिए, फीफा में इस्तेमाल होने वाले फुटबॉल का वजन **410 ग्राम से 450 ग्राम** के बीच होता है।


In [ ]:
# Gemma

import torch
from transformers import AutoTokenizer, Gemma3ForCausalLM

custom_cache_dir = "/scratch/bbjr/skarmakar/huggingface"

model_id = "google/gemma-3-12b-it"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = Gemma3ForCausalLM.from_pretrained(
    model_id, 
    device_map="auto", 
    torch_dtype=torch.bfloat16,
    # torch_dtype=torch.float16,
    cache_dir=custom_cache_dir,
    
)

model.generation_config.temperature=None # to disable the stupid warnings
model.generation_config.top_p=None # to disable the stupid warnings
model.generation_config.top_k=None # to disable the stupid warnings

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

In [33]:
messages = [
    # {"role": "system", "content": "You are a helpful assistant."},

    # {"role": "user", "content": "What is weight of a football used in fifa?"}
    # {"role": "user", "content": "फीफा में इस्तेमाल होने वाली फुटबॉल का वज़न कितना होता है?"}
    {"role": "user", "content": [{"type": "text", "text": "What is weight of a football used in fifa?"},]}
]

# # Apply the template and move to device
# inputs = tokenizer.apply_chat_template(
#     messages, 
#     add_generation_prompt=True, 
#     return_tensors="pt"
# ).to(model.device)

# outputs = model.generate(inputs, max_new_tokens=256, do_sample=False)
# print(tokenizer.decode(outputs[0], skip_special_tokens=True))

prompts = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
# prompts = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True).strip()

out = tokenizer(prompts, return_tensors='pt', padding=True, add_special_tokens=False).to(model.device)

outputs = model.generate(**out, max_new_tokens=256, do_sample=False)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

user
What is weight of a football used in fifa?
model
The official weight of a FIFA-approved football (soccer ball) is quite specific:

*   **Between 410 and 450 grams (14.5 to 15.9 ounces)**

This is a requirement outlined in the FIFA Quality Programme for match balls.



There's also a size requirement: circumference between 68 and 70 cm (26.8 to 27.6 inches).


Comparisions

In [ ]:
lang2 = "hindi"
lang1 = "italian"
lang3 = "english"


l1_controller = load_controller_translate(llm, lang1, lang2, path=f'../all_gitignore/language_multi/')
l2_controller = load_controller_translate(llm, lang1, lang3, path=f'../all_gitignore/language_multi/')

l3_controller = load_controller_translate(llm, lang3, lang1, path=f'../all_gitignore/language_multi/')
l4_controller = load_controller_translate(llm, lang2, lang1, path=f'../all_gitignore/language_multi/')



In [ ]:
compare_cosine(l1_controller.directions, l2_controller.directions)

layer: -1, cosine: 0.3777655065059662
layer: -2, cosine: 0.3728252053260803
layer: -3, cosine: 0.3848298490047455
layer: -4, cosine: 0.41419360041618347
layer: -5, cosine: 0.4364524483680725
layer: -6, cosine: 0.37483206391334534
layer: -7, cosine: 0.2924160361289978
layer: -8, cosine: 0.33504343032836914
layer: -9, cosine: 0.4393605887889862
layer: -10, cosine: 0.4389570653438568
layer: -11, cosine: 0.45833808183670044
layer: -12, cosine: 0.4171558618545532
layer: -13, cosine: 0.4602075219154358
layer: -14, cosine: 0.4546092748641968
layer: -15, cosine: 0.43455761671066284
layer: -16, cosine: 0.2203870266675949
layer: -17, cosine: 0.3391492962837219
layer: -18, cosine: 0.2877112627029419
layer: -19, cosine: 0.37672093510627747
layer: -20, cosine: 0.3937227725982666
layer: -21, cosine: 0.38694682717323303
layer: -22, cosine: 0.34846439957618713
layer: -23, cosine: 0.3627947270870209
layer: -24, cosine: 0.4781742990016937
layer: -25, cosine: 0.46991828083992004
layer: -26, cosine: 0.576

0.417615897713169

In [ ]:
compare_cosine(l3_controller.directions, l4_controller.directions)

layer: -1, cosine: 0.03683067858219147
layer: -2, cosine: -0.10372020304203033
layer: -3, cosine: -0.042768292129039764
layer: -4, cosine: 0.039587926119565964
layer: -5, cosine: 0.015600902959704399
layer: -6, cosine: 0.020478736609220505
layer: -7, cosine: -0.19523707032203674
layer: -8, cosine: -0.2240738421678543
layer: -9, cosine: -0.2050919234752655
layer: -10, cosine: -0.16165439784526825
layer: -11, cosine: 0.06776690483093262
layer: -12, cosine: -0.025502612814307213
layer: -13, cosine: -0.051080696284770966
layer: -14, cosine: -0.059766191989183426
layer: -15, cosine: -0.004298009444028139
layer: -16, cosine: 0.0592011883854866
layer: -17, cosine: 0.018897827714681625
layer: -18, cosine: 0.06311559677124023
layer: -19, cosine: 0.0029665473848581314
layer: -20, cosine: 0.16803759336471558
layer: -21, cosine: 0.33914434909820557
layer: -22, cosine: 0.10224539041519165
layer: -23, cosine: 0.021879443898797035
layer: -24, cosine: 0.02176785096526146
layer: -25, cosine: 0.03555804

0.024803640335918434

Flores

In [11]:
from datasets import load_dataset

In [21]:
ds_orig = load_dataset("facebook/flores", "eng_Latn", split="dev", trust_remote_code=True)
ds_other = load_dataset("facebook/flores", "hin_Deva", split="dev", trust_remote_code=True)

In [18]:
print(ds_orig)
print(ds_orig[10])

Dataset({
    features: ['id', 'URL', 'domain', 'topic', 'has_image', 'has_hyperlink', 'sentence'],
    num_rows: 997
})
{'id': 11, 'URL': 'https://en.wikinews.org/wiki/Thousands_march_in_London_calling_for_David_Cameron%27s_resignation_over_tax_affairs', 'domain': 'wikinews', 'topic': 'politics', 'has_image': 0, 'has_hyperlink': 0, 'sentence': 'Around 11:29, the protest moved up Whitehall, past Trafalgar Square, along the Strand, passing by Aldwych and up Kingsway towards Holborn where the Conservative Party were holding their Spring Forum in the Grand Connaught Rooms hotel.'}


In [19]:
print(ds_other)
print(ds_other[10])

Dataset({
    features: ['id', 'URL', 'domain', 'topic', 'has_image', 'has_hyperlink', 'sentence'],
    num_rows: 997
})
{'id': 11, 'URL': 'https://en.wikinews.org/wiki/Thousands_march_in_London_calling_for_David_Cameron%27s_resignation_over_tax_affairs', 'domain': 'wikinews', 'topic': 'politics', 'has_image': 0, 'has_hyperlink': 0, 'sentence': '11:29 बजे के आस-पास, विरोध करने वाले लोग व्हाइटहॉल की ओर बढ़ गए. इसके बाद, वे किनारे-किनारे चलते हुए ट्राफ़ल्गर स्क्वायर, एल्डविच, किंग्सवे से गुजरते हुए होलबर्न की ओर बढ़ गए, जहाँ कंजरवेटिव पार्टी, ग्रैंड कनॉट रूम होटल में अपने स्प्रिंग फ़ोरम का आयोजन कर रही थी.'}
